# 02 — Pré-processamento Visual, Controle de Qualidade e Divisão Estratificada (70/15/15)

Atende às Seções **3.3, 3.4 e 4.2** da documentação técnica:
1. Verificação de imagens corrompidas.
2. Identificação de arquivos duplicados (prevenção de *data leakage*).
3. Padronização das dimensões (`224x224`) e normalização (`[0.0, 1.0]`).
4. Extração de 30 características visuais (índices RGB/ExG, área necrótica, halo clorótico, pústulas e textura).
5. Separação estratificada em **70% Treinamento / 15% Validação / 15% Teste**.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.vision.image_processor import ImageProcessor
from src.ml.training import VisualModelTrainer

processor = ImageProcessor()
trainer = VisualModelTrainer(random_state=42)
dataset = trainer.build_and_split_dataset(samples_per_class=100, inject_corrupted=4, inject_duplicates=6)
print(json.dumps(dataset.summary.to_dict(), indent=2, ensure_ascii=False))

{
  "total_raw_images": 610,
  "corrupted_removed": 4,
  "duplicates_removed": 6,
  "valid_unique_images": 600,
  "train_count": 420,
  "val_count": 90,
  "test_count": 90,
  "proportions": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "class_distribution_train": {
    "Corn_(maize)___Common_rust_": 70,
    "Potato___Early_blight": 70,
    "Potato___Late_blight": 70,
    "Tomato___Early_blight": 70,
    "Tomato___Healthy": 70,
    "Tomato___Late_blight": 70
  },
  "class_distribution_val": {
    "Corn_(maize)___Common_rust_": 15,
    "Potato___Early_blight": 15,
    "Potato___Late_blight": 15,
    "Tomato___Early_blight": 15,
    "Tomato___Healthy": 15,
    "Tomato___Late_blight": 15
  },
  "class_distribution_test": {
    "Corn_(maize)___Common_rust_": 15,
    "Potato___Early_blight": 15,
    "Potato___Late_blight": 15,
    "Tomato___Early_blight": 15,
    "Tomato___Healthy": 15,
    "Tomato___Late_blight": 15
  }
}


In [2]:
# Inspeção de qualidade em imagem nítida vs. imagem escura vs. imagem borrada
for mode in ['Tomato___Early_blight', 'dark', 'blurry']:
    img = ImageProcessor.create_synthetic_leaf_image(mode)
    res = processor.process(img)
    print(f'Modo={mode:<24} | válida={str(res.is_valid):<5} | brilho={res.brightness:6.1f} | nitidez={res.blur_score:7.1f} | padrões={res.detected_visual_patterns}')

Modo=Tomato___Early_blight    | válida=True  | brilho= 120.4 | nitidez= 1916.0 | padrões=['lamina_foliar_verde_homogenea']
Modo=dark                     | válida=False | brilho=   6.5 | nitidez=   75.6 | padrões=['padrao_foliar_geral']
Modo=blurry                   | válida=False | brilho= 110.0 | nitidez=    0.0 | padrões=['padrao_foliar_geral']
